In [ ]:
from ollama import chat

response = chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "user", "content": "Hello"}
    ]
)

print(response["message"]["content"])

ResponseError: an error was encountered while running the model: error: task id = 0, error: got exception: std::bad_alloc (status code: 500)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
1
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).cuda()

print(next(model.parameters()).device)

d:\OneDrive\Documents\GitHub\summer-intern-2026-02\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [4]:
from ollama import chat

response = chat(
    model="qwen3.5:4b",
    messages=[
        {
            "role": "system",
            "content": "You are an expert text OCR reader. You will be given an image I want you to return its content as Markdown"
        },
        {
            "role": "user",
            "images": ["image.png"]
        }
      ],
    think=False
)

print(response["message"]["content"])

ResponseError: llama-server startup failed before projector CPU offload retry: llama-server reported out-of-memory during startup: ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 774313984
alloc_tensor_range: failed to allocate CPU buffer of size 774313984
GGML_ASSERT(buffer); error stopping failed process: TerminateProcess: Access is denied. (status code: 500)

In [8]:
!pip install pymupdf

In [9]:
import fitz
import os

pdf_path = "1_README.pdf" 
output_folder = "pages"

os.makedirs(output_folder, exist_ok=True)

doc = fitz.open(pdf_path)

for page_number in range(len(doc)):
    page = doc[page_number]

    pix = page.get_pixmap(dpi=300)

    image_path = os.path.join(
        output_folder,
        f"page_{page_number + 1}.png"
    )

    pix.save(image_path)

    print(f"Saved: {image_path}")

doc.close()



Saved: pages\page_1.png
Saved: pages\page_2.png
Saved: pages\page_3.png


In [10]:
!pip install ollama

In [ ]:
from ollama import chat

response = chat(
    model="qwen3.5:4b",
    messages=[
        {
            "role": "system",
            "content": """
You are an OCR engine.
Extract all text from the image exactly.
Return only Markdown.
Do not summarize.
Do not stop until you finish the whole page.
"""
        },
        {
            "role": "user",
            "images": ["pages/page_1.png"]
        }
    ],
    options={
        "num_ctx": 16384,
        "num_predict": -1
    },
    think=False
)

print(response["message"]["content"])

In [1]:
from ollama import chat
import os

images_folder = "pages"
output_file = "document.md"

markdown_content = ""

images = sorted(os.listdir(images_folder))

for image in images:
    image_path = os.path.join(images_folder, image)

    print(f"Processing {image}...")

    response = chat(
        model="qwen3.5:4b",
        messages=[
            {
                "role": "system",
                "content": """
You are an OCR engine.
Extract all text from the image exactly.
Return only Markdown.
Do not summarize.
Do not stop until you finish the whole page.
"""
            },
            {
                "role": "user",
                "images": [image_path]
            }
        ],
        options={
            "num_ctx": 16384,
            "num_predict": -1
        },
        think=False
    )

    page_markdown = response["message"]["content"]

    markdown_content += f"\n\n<!-- {image} -->\n\n"
    markdown_content += page_markdown


with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)


print("Done! Saved:", output_file)

Processing page_1.png...
Processing page_2.png...
Processing page_3.png...
Done! Saved: document.md


In [2]:
pip install datasets huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import load_dataset

dataset = load_dataset("v1v1d/v1v1d_docmatix_test_v1_fast")

print(dataset)

d:\OneDrive\Documents\GitHub\summer-intern-2026-02\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'pdf_name', 'page_number', 'page_size', 'layout', 'content_list', 'middle', 'model', 'markdown', 'html', 'html_with_coordinates', 'html_compact', 'raw_md', 'lines', 'images', 'equations', 'tables', 'pdf_info', 'vqa'],
        num_rows: 1500
    })
})


In [2]:
sample = dataset["train"][0]

print(sample)

{'id': '00006_page_0000', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1993x2579 at 0x132970C0C90>, 'pdf_name': '00006', 'page_number': 0, 'page_size': [956, 1237], 'layout': [{'type': 'title', 'coordinates': [166, 215, 867, 288], 'content': 'PUZZLE MADNESS', 'index': 0}, {'type': 'text', 'coordinates': [121, 325, 900, 355], 'content': 'Daily Medium Mathdoku Puzzle for Friday 18th June 2021', 'index': 1}, {'type': 'image_body', 'coordinates': [322, 382, 701, 675], 'content': '', 'index': 2}, {'type': 'discarded', 'coordinates': [432, 76, 598, 202], 'content': '', 'index': 3}], 'content_list': '[{"type": "text", "text": "PUZZLE MADNESS ", "text_level": 1, "page_idx": 0}, {"type": "text", "text": "Daily Medium Mathdoku Puzzle for Friday 18th June 2021 ", "page_idx": 0}, {"type": "image", "img_path": "images/11494ec3b1bce232bc8b9e151013523734d29142cd27657caeeb5ce3cfe5128a.jpg", "img_caption": [], "img_footnote": [], "page_idx": 0}]', 'middle': '{"preproc_blocks": [{"type"

In [3]:
for key, value in sample.items():
    print("=" * 50)
    print(key)
    print(type(value))
    

id
<class 'str'>
image
<class 'PIL.PngImagePlugin.PngImageFile'>
pdf_name
<class 'str'>
page_number
<class 'int'>
page_size
<class 'list'>
layout
<class 'list'>
content_list
<class 'str'>
middle
<class 'str'>
model
<class 'str'>
markdown
<class 'str'>
html
<class 'str'>
html_with_coordinates
<class 'str'>
html_compact
<class 'str'>
raw_md
<class 'str'>
lines
<class 'str'>
images
<class 'str'>
equations
<class 'str'>
tables
<class 'str'>
pdf_info
<class 'str'>
vqa
<class 'list'>


In [4]:
sample = dataset["train"][1233]

print(sample["raw_md"])

# 12 Dust Control  

ust controls may be needed on industrial sites for various reasons, including land disturbance, demolition and material handling areas. If effective, dust controls can prevent pollutants from contaminating stormwater runoff by reducing the surface and air transport of dust caused by these activities.  

# Practices to ControlDustfrom Land Disturbances and DemolitionActivities  

$\Cup$ Use temporary controls, such as palliatives or chemical soil treatments, that are applied as sprayon adhesives. Common palliatives include:  

$\nu$ Calcium Chloride   
$\nu$ Ammoniac Asphalt Emulsion   
VLatexEmulsion   
$\nu$ Resin-Water Emulsion   
VLignin  

$\circleddash$ Since certain chemicals may be inappropriate for some soil types or application areas, the permittee should check with the City and Department of Environmental Quality (DEQ) prior to the application of chemical treatments. Vehiclesshouldnotbe driven over the treated area to avoid the tracking of the chemicals t

In [5]:
print(sample["markdown"])

# 12 Dust Control

![img_1](546,163,966,316)

![img_2](546,163,966,316)

ust controls may be needed on industrial sites for various reasons, including land distur- bance, demolition and material handling areas. If effective, dust controls can prevent pollutants from contaminating stormwater runoff by reducing the surface and air transport of dust caused by these activities.

soil exposure by temporary or permanent soil sta- bilization controls, such as:

# Practices to ControlDustfrom Land Disturbances and DemolitionActivities

- Mulching /Seeding $\nu$ Spreading coarse gravel or crushed stone $\nu$ Planting trees

$\Cup$ Use temporary controls, such as palliatives or chemical soil treatments, that are applied as spray- on adhesives. Common palliatives include:

$\mathbf{\^{ep}}$  Install temporary or permanent windbreaks or barriers that reduce airborne particles by slowing wind velocities and causing particles to drop. Large trees and shrubs left in place can provide wind barriers, w

In [6]:
sample = dataset["train"][0]

sample["image"].save("test.png")

with open("test.md", "w", encoding="utf-8") as f:
    f.write(sample["markdown"])

print("Done")

Done


In [7]:
print(len(dataset["train"]))

1500


In [8]:
small_dataset = dataset["train"].select(range(10))

def preprocess(example):
    return {
        "image": example["image"],
        "prompt": "Convert this document to Markdown.",
        "answer": example["markdown"]
    }

train_dataset = small_dataset.map(
    preprocess,
    remove_columns=small_dataset.column_names
)

print(train_dataset)

Dataset({
    features: ['image', 'prompt', 'answer'],
    num_rows: 10
})


In [9]:
print(train_dataset[0]["prompt"])
print("=" * 50)
print(train_dataset[0]["answer"][:500])

Convert this document to Markdown.
# PUZZLE MADNESS

Daily Medium Mathdoku Puzzle for Friday 18th June 2021

![img_1](322,382,701,675)

![img_2](322,382,701,675)



In [ ]:
from IPython.display import display

display(train_dataset[0]["image"])

In [11]:
import torch

print(torch.__version__)
print(torch.version.cuda)

2.6.0+cu124
12.4


In [12]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [9]:
import transformers
import peft
import trl

print(transformers.__version__)
print(peft.__version__)
print(trl.__version__)

5.14.1
0.19.1
1.9.0


In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Model Loaded")

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 995.53it/s]


Model Loaded


In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 7,744,512 || all params: 264,229,440 || trainable%: 2.9310


In [6]:
from datasets import load_dataset

dataset = load_dataset("v1v1d/v1v1d_docmatix_test_v1_fast")

print(dataset)
print(dataset["train"][0].keys())

DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'pdf_name', 'page_number', 'page_size', 'layout', 'content_list', 'middle', 'model', 'markdown', 'html', 'html_with_coordinates', 'html_compact', 'raw_md', 'lines', 'images', 'equations', 'tables', 'pdf_info', 'vqa'],
        num_rows: 1500
    })
})
dict_keys(['id', 'image', 'pdf_name', 'page_number', 'page_size', 'layout', 'content_list', 'middle', 'model', 'markdown', 'html', 'html_with_coordinates', 'html_compact', 'raw_md', 'lines', 'images', 'equations', 'tables', 'pdf_info', 'vqa'])


In [10]:
train_dataset = dataset["train"].select(range(150))

def format_example(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {
                        "type": "text",
                        "text": "Convert this document image to Markdown."
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": example["markdown"]
                    }
                ]
            }
        ],
        "image": example["image"]
    }

train_dataset = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names
)

In [11]:
print(train_dataset[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1993x2579 at 0x19AE7F2AB50>, 'messages': [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'Convert this document image to Markdown.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '# PUZZLE MADNESS\n\nDaily Medium Mathdoku Puzzle for Friday 18th June 2021\n\n![img_1](322,382,701,675)\n\n![img_2](322,382,701,675)\n'}]}]}


In [12]:
from PIL import Image

IGNORE_INDEX = -100

def collate_fn(examples):

    texts = []
    images = []

    for example in examples:

        text = processor.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)
        images.append(example["image"])

    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()

    assistant_token_ids = processor.tokenizer.encode(
        "Assistant:",
        add_special_tokens=False
    )

    for i in range(labels.size(0)):

        ids = labels[i].tolist()

        start = None

        for j in range(len(ids) - len(assistant_token_ids)):
            if ids[j:j+len(assistant_token_ids)] == assistant_token_ids:
                start = j + len(assistant_token_ids)
                break

        if start is None:
            labels[i][:] = IGNORE_INDEX
        else:
            labels[i][:start] = IGNORE_INDEX

        labels[i][labels[i] == processor.tokenizer.pad_token_id] = IGNORE_INDEX

    batch["labels"] = labels

    return batch

In [7]:
batch = collate_fn([train_dataset[0]])

print(processor.tokenizer.decode(batch["labels"][0][batch["labels"][0] != -100]))

 # PUZZLE MADNESS

Daily Medium Mathdoku Puzzle for Friday 18th June 2021

![img_1](322,382,701,675)

![img_2](322,382,701,675)
<end_of_utterance>



In [13]:
processor.image_processor.do_image_splitting = False

processor.image_processor.size = {
    "height": 384,
    "width": 384
}

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./smolvlm-ocr",

    # Batch
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # Training
    num_train_epochs=2,
    learning_rate=2e-4,

    # Memory
    fp16=True,
    gradient_checkpointing=True,

    # Logging
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,

    # DataLoader
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

In [15]:
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    data_collator=collate_fn,
)

In [16]:
print(model)
print(processor)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Idefics3ForConditionalGeneration(
      (model): Idefics3Model(
        (vision_model): Idefics3VisionTransformer(
          (embeddings): Idefics3VisionEmbeddings(
            (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
            (position_embedding): Embedding(1024, 768)
          )
          (encoder): Idefics3Encoder(
            (layers): ModuleList(
              (0-11): 12 x Idefics3EncoderLayer(
                (self_attn): Idefics3VisionAttention(
                  (k_proj): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                   

In [17]:
print(len(train_dataset))
print(train_dataset[0].keys())

150
dict_keys(['image', 'messages'])


In [18]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    data_collator=collate_fn,
)

print("Trainer Ready")

Trainer Ready


In [19]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49279, 'bos_token_id': 1, 'pad_token_id': 2}.


Step,Training Loss
10,3.115489
20,2.327859
30,2.569694


TrainOutput(global_step=38, training_loss=2.632912886770148, metrics={'train_runtime': 295.9105, 'train_samples_per_second': 1.014, 'train_steps_per_second': 0.128, 'total_flos': 229184105492736.0, 'train_loss': 2.632912886770148, 'entropy': 2.446228584935588, 'num_tokens': 162502.0, 'mean_token_accuracy': 0.5547224206309165, 'epoch': 2.0})

In [20]:
model.save_pretrained("./smolvlm-ocr-lora")
processor.save_pretrained("./smolvlm-ocr-lora")

['./smolvlm-ocr-lora\\processor_config.json']

In [21]:
image = train_dataset[0]["image"]

In [24]:
import torch

# خلي الموديل في وضع التقييم
model.eval()

# أول صورة من الـ dataset
image = train_dataset[0]["image"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": "Convert this document image to Markdown."
            }
        ]
    }
]

# إنشاء الـ Prompt
prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# تجهيز الـ Inputs
inputs = processor(
    text=prompt,
    images=[image],
    return_tensors="pt"
).to(model.device)

# توليد الإجابة
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        early_stopping=True,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

# حذف الـ Prompt والإبقاء على الإجابة فقط
generated_ids = outputs[:, inputs["input_ids"].shape[1]:]

prediction = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("=" * 80)
print("Prediction:")
print(prediction)

print("\n" + "=" * 80)
print("Ground Truth:")
print(train_dataset[0]["messages"][1]["content"][0]["text"])

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prediction:
 PUZZLEMADNESS
Daily Medium MathodokuPuzzle for Friday 18th June2021

#PUZZLEMADNESS

##

###

####

**PUZZLEMADNESS**

- **Daily Medium Mathodoku Puzzle for Friday 18th June2021**

---

The Daily Medium Mathodoku Puzzle is a puzzle that involves solving a math problem using the

Ground Truth:
# PUZZLE MADNESS

Daily Medium Mathdoku Puzzle for Friday 18th June 2021

![img_1](322,382,701,675)

![img_2](322,382,701,675)



In [9]:
batch = collate_fn([train_dataset[0]])

for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([1, 160])
attention_mask torch.Size([1, 160])
pixel_values torch.Size([1, 1, 3, 512, 512])
pixel_attention_mask torch.Size([1, 1, 512, 512])
labels torch.Size([1, 160])


In [10]:
print(processor.image_processor)

Idefics3ImageProcessorPil {
  "do_convert_rgb": true,
  "do_image_splitting": false,
  "do_normalize": true,
  "do_pad": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Idefics3ImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "max_image_size": {
    "longest_edge": 512
  },
  "resample": 1,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 384,
    "width": 384
  }
}



In [7]:
from trl import SFTTrainer

print("TRL Ready")

TRL Ready


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./smolvlm-ocr",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none"
)

print("TrainingArguments Updated")

TrainingArguments Updated


In [9]:
from PIL import Image
import io

def collate_fn(examples):

    texts = []
    images = []

    for example in examples:

        text = processor.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )

        texts.append(text)

        img = example["images"][0]

        if img["bytes"] is not None:
            image = Image.open(
                io.BytesIO(img["bytes"])
            ).convert("RGB")
        else:
            image = Image.open(
                img["path"]
            ).convert("RGB")

        images.append(image)


    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    batch["labels"] = batch["input_ids"].clone()

    return batch

In [10]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    data_collator=collate_fn,
)

print("Trainer Updated")

Trainer Updated


In [11]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49279, 'bos_token_id': 1, 'pad_token_id': 2}.


Step,Training Loss
5,13.160709
10,9.059842
15,4.186521
20,1.556556
25,1.238267
30,1.051751
35,0.971220


TrainOutput(global_step=38, training_loss=4.190597753775747, metrics={'train_runtime': 1534.6118, 'train_samples_per_second': 0.195, 'train_steps_per_second': 0.025, 'total_flos': 617288860424448.0, 'train_loss': 4.190597753775747, 'entropy': 0.890768133781173, 'num_tokens': 437686.0, 'mean_token_accuracy': 0.8419012156399813, 'epoch': 2.0})

In [11]:
print(train_dataset)

Dataset({
    features: ['id', 'image', 'pdf_name', 'page_number', 'page_size', 'layout', 'content_list', 'middle', 'model', 'markdown', 'html', 'html_with_coordinates', 'html_compact', 'raw_md', 'lines', 'images', 'equations', 'tables', 'pdf_info', 'vqa', 'messages'],
    num_rows: 200
})


In [12]:
batch = collate_fn(
    [train_dataset[0]]
)

for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([1, 1220])
attention_mask torch.Size([1, 1220])
pixel_values torch.Size([1, 17, 3, 512, 512])
pixel_attention_mask torch.Size([1, 17, 512, 512])
labels torch.Size([1, 1220])


In [13]:
batch = collate_fn([train_dataset[0]])

# نحط البيانات على نفس الجهاز بتاع الموديل
batch = {
    k: v.to(model.device)
    for k, v in batch.items()
}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.loss)

tensor(0.2347, device='cuda:0')


In [14]:
model.print_trainable_parameters()

trainable params: 7,744,512 || all params: 264,229,440 || trainable%: 2.9310


In [15]:
model.save_pretrained("./smolvlm-ocr-lora")
processor.save_pretrained("./smolvlm-ocr-lora")

print("Model Saved")

Model Saved


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

base_model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

processor = AutoProcessor.from_pretrained(base_model_id)

base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

model = PeftModel.from_pretrained(
    base_model,
    "./smolvlm-ocr-lora"
)

model.eval()

print("Model Loaded")

Loading weights: 100%|██████████| 471/471 [00:00<00:00, 500.82it/s]


Model Loaded


In [19]:
import os

print(os.getcwd())

d:\OneDrive\Documents\GitHub\summer-intern-2026-02\Abdalah anwer


In [21]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".png"):
            print(os.path.join(root, file))

.\image-flower.png
.\image.png
.\test.png
.\pages\page_1.png
.\pages\page_2.png
.\pages\page_3.png


In [22]:
import io
from PIL import Image

example = train_dataset[0]
img = example["images"][0]

if img["bytes"] is not None:
    image = Image.open(io.BytesIO(img["bytes"])).convert("RGB")
else:
    image = Image.open(img["path"]).convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Convert this document to Markdown."}
        ]
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=prompt,
    images=[image],
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.2,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

generated_text = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(generated_text)

 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  | 


In [23]:
print(train_dataset[0]["messages"])
print(train_dataset[0]["images"])

[{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'Convert this document image to Markdown.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '# PUZZLE MADNESS\n\nDaily Medium Mathdoku Puzzle for Friday 18th June 2021\n\n![img_1](322,382,701,675)\n\n![img_2](322,382,701,675)\n'}]}]
[{'path': None, 'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x07\xc9\x00\x00\n\x13\x08\x02\x00\x00\x008J\xf4\xd6\x00\x01\x00\x00IDATx\x9c\xec\xdd\xe9s\\\xe7\x95\'h\xe4\xbe"\xb1\x83\x00wR\x14I\x91\xb2$[\x96-\xd9\xaary\xa9ru\xb8\xabbb"\xa6>\xce\x7f7\xd1\x9f\xa6\xa3\xa2g\xbak\xda\xdd\xaerYRY\xd6.\x8a+\xb8\x81\x04\x08\x80\xd8\x91\xfbzs\xc2\xb8v6\x8b\xf2\xc2\x94\x08\x02 \x9e\xe7\x83\x82\xba\x002\xdf|\x91\x00n\xfe\xee\xc9s"\xddnw\x00\x00\x00\x00\x00\x00xb\xd1\'\xffT\x00\x00\x00\x00\x00@\xb6\x0e\x00\x00\x00\x00\x00}S\xb7\x0e\x00\x00\x00\x00\x00\xfd\x91\xad\x03\x00\x00\x00\x00@\x7fd\xeb\x00\x00\x00\x00\x00\xd0\x1f\xd9:\x00\x00\x00\x00\x00\xf4G\xb6\x0e\x00\x00\x00\x00\x

In [24]:
text = processor.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)

print(text)

<|im_start|>User:<image>Convert this document image to Markdown.<end_of_utterance>
Assistant: # PUZZLE MADNESS

Daily Medium Mathdoku Puzzle for Friday 18th June 2021

![img_1](322,382,701,675)

![img_2](322,382,701,675)
<end_of_utterance>



In [25]:
batch = collate_fn([train_dataset[0]])

print(processor.tokenizer.decode(batch["labels"][0]))

<|im_start|>User:<fake_token_around_image><row_1_col_1><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><fake_token_around_image><row_1_col_2><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><fake_token

In [26]:
import transformers
import trl

print(transformers.__version__)
print(trl.__version__)

5.14.1
1.9.0


In [18]:
model = PeftModel.from_pretrained(
    base_model,
    "./smolvlm-ocr-lora"
)

d:\OneDrive\Documents\GitHub\summer-intern-2026-02\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:
print(type(model))
print(model)

<class 'peft.peft_model.PeftModelForCausalLM'>
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Idefics3ForConditionalGeneration(
      (model): Idefics3Model(
        (vision_model): Idefics3VisionTransformer(
          (embeddings): Idefics3VisionEmbeddings(
            (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
            (position_embedding): Embedding(1024, 768)
          )
          (encoder): Idefics3Encoder(
            (layers): ModuleList(
              (0-11): 12 x Idefics3EncoderLayer(
                (self_attn): Idefics3VisionAttention(
                  (k_proj): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=F

In [21]:
from PIL import Image
import io

example = train_dataset[0]

img = example["images"][0]

if img["bytes"] is not None:
    image = Image.open(io.BytesIO(img["bytes"])).convert("RGB")
else:
    image = Image.open(img["path"]).convert("RGB")

print(example["messages"][1]["content"][0]["text"])

# PUZZLE MADNESS

Daily Medium Mathdoku Puzzle for Friday 18th June 2021

![img_1](322,382,701,675)

![img_2](322,382,701,675)



In [26]:
from pprint import pprint

sample = train_dataset[0]

pprint(sample)

{'content_list': '[{"type": "text", "text": "PUZZLE MADNESS ", "text_level": '
                 '1, "page_idx": 0}, {"type": "text", "text": "Daily Medium '
                 'Mathdoku Puzzle for Friday 18th June 2021 ", "page_idx": 0}, '
                 '{"type": "image", "img_path": '
                 '"images/11494ec3b1bce232bc8b9e151013523734d29142cd27657caeeb5ce3cfe5128a.jpg", '
                 '"img_caption": [], "img_footnote": [], "page_idx": 0}]',
 'equations': '[]',
 'html': '<div class="pdf-page">\n'
         '<h1>PUZZLE MADNESS</h1>\n'
         '<p>Daily Medium Mathdoku Puzzle for Friday 18th June 2021</p>\n'
         '<figure><img data-bbox="322,382,701,675" />\n'
         '</figure>\n'
         '<figure><img data-bbox="322,382,701,675" />\n'
         '</figure>\n'
         '</div>',
 'html_compact': '<div class="page">\n'
                 '<h1 class="title" bbox="166,215,701,73">PUZZLE MADNESS</h1>\n'
                 '<p class="text" bbox="121,325,779,30">Daily Medium M

In [27]:
print(train_dataset.column_names)

['id', 'image', 'pdf_name', 'page_number', 'page_size', 'layout', 'content_list', 'middle', 'model', 'markdown', 'html', 'html_with_coordinates', 'html_compact', 'raw_md', 'lines', 'images', 'equations', 'tables', 'pdf_info', 'vqa', 'messages']


In [29]:
print(len(train_dataset))

100


In [26]:
print(type(train_dataset[0]["images"]))
print(type(train_dataset[0]["images"][0]))

<class 'list'>
<class 'dict'>


In [27]:
print(train_dataset[0]["images"][0].keys())

dict_keys(['path', 'bytes'])


In [39]:
processor.image_processor.size = {
    "height": 384,
    "width": 384
}

In [43]:
processor.image_processor.max_image_size = {
    "longest_edge": 384
}

In [44]:
batch = collate_fn([train_dataset[0]])

for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([1, 160])
attention_mask torch.Size([1, 160])
pixel_values torch.Size([1, 1, 3, 384, 384])
pixel_attention_mask torch.Size([1, 1, 384, 384])
labels torch.Size([1, 160])


In [45]:
model.gradient_checkpointing_enable()